# ROE - Re_5y Spread Analysis
FMP API에서 분기별 ROE를 수신하여 Re_5y - ROE 스프레드를 측정합니다.

## 0. 경로 설정 (노트북 / 데스크탑 자동 감지)

In [6]:
import sys
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pymysql
from typing import Dict, List, Optional

NOTEBOOK_DATA_PATH = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast\DATA"
DESKTOP_DATA_PATH  = r"/DATA"

for _path in [NOTEBOOK_DATA_PATH, DESKTOP_DATA_PATH]:
    if os.path.isdir(_path) and _path not in sys.path:
        sys.path.insert(0, _path)
        print(f"[PATH] {_path} 추가됨")
        break

from DATA.config import get_db_info, get_engine
from DATA.us_target_ticker_list_2000 import ticker_list as US_TICKER_LIST

## 1. DB 설정

In [8]:
db_info = get_db_info()
engine  = get_engine(db_info)
print("[DB] 연결 정보 로드 완료")

[DB] 연결 정보 로드 완료


## 2. FMP API 설정 및 ROE 수신 함수

In [9]:
FMP_API_KEY = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"   # ← 실제 키로 교체


def fetch_fmp_roe(ticker: str, api_key: str = FMP_API_KEY, limit: int = 60) -> pd.DataFrame:
    """
    FMP Financial Ratios (분기별) 엔드포인트에서 ROE를 가져온다.
    반환: date | ticker | roe  (소수점, 예: 0.15 = 15%)
    """
    url = (
        f"https://financialmodelingprep.com/api/v3/ratios/{ticker}"
        f"?period=quarter&limit={limit}&apikey={api_key}"
    )
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    data = resp.json()

    if not data:
        print(f"[WARN] FMP ROE: {ticker} 데이터 없음")
        return pd.DataFrame(columns=["date", "ticker", "roe"])

    df = pd.DataFrame(data)[["date", "returnOnEquity"]].copy()
    df.rename(columns={"returnOnEquity": "roe"}, inplace=True)
    df["ticker"] = ticker
    df["date"]   = pd.to_datetime(df["date"])
    df["roe"]    = pd.to_numeric(df["roe"], errors="coerce")
    df.sort_values("date", inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df[["date", "ticker", "roe"]]

## 3. DB에서 Re_5y pivot 조회 함수

In [10]:
def fetch_re5y_from_db(
    db_info: Dict,
    tickers: Optional[List[str]] = None,
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    table: str = "us_required_return_result",
) -> pd.DataFrame:
    """
    us_required_return_result 테이블에서 Re_5y를 일별로 가져온 후
    월말 기준으로 resample.
    반환: date | ticker | Re_5y
    """
    conn = pymysql.connect(
        host=db_info["host"], port=db_info["port"],
        user=db_info["user"], password=db_info["password"],
        database=db_info["database"], charset="utf8mb4"
    )
    where_clauses = ["indicator = %s"]
    params = ["Re_5y"]

    if start_date:
        where_clauses.append("date >= %s"); params.append(start_date)
    if end_date:
        where_clauses.append("date <= %s"); params.append(end_date)
    if tickers:
        ph = ", ".join(["%s"] * len(tickers))
        where_clauses.append(f"ticker IN ({ph})"); params.extend(tickers)

    where_sql = "WHERE " + " AND ".join(where_clauses)
    query = f"""
        SELECT date, ticker, value AS Re_5y
        FROM {table}
        {where_sql}
        ORDER BY date, ticker;
    """
    try:
        df = pd.read_sql(query, conn, params=params)
    finally:
        conn.close()

    df["date"] = pd.to_datetime(df["date"])

    # 월말 기준 집계 (각 월의 마지막 거래일)
    df["month_end"] = df["date"] + pd.offsets.MonthEnd(0)
    df_monthly = (
        df.groupby(["ticker", "month_end"])
          .last()
          .reset_index()
          .drop(columns=["date"])
          .rename(columns={"month_end": "date"})
    )
    return df_monthly[["date", "ticker", "Re_5y"]]

## 4. ROE 월말 보간 (분기 → 월별 ffill)

In [11]:
def roe_to_monthly(roe_df: pd.DataFrame) -> pd.DataFrame:
    """
    FMP 분기 ROE → 월말 ffill (limit=2, 즉 3개월 간격)
    """
    roe_df = roe_df.copy()
    roe_df["date"] = pd.to_datetime(roe_df["date"])
    roe_df["month_end"] = roe_df["date"] + pd.offsets.MonthEnd(0)
    roe_df = roe_df.drop(columns=["date"]).rename(columns={"month_end": "date"})

    roe_start = roe_df["date"].min()
    roe_end   = roe_df["date"].max()
    all_me    = pd.date_range(start=roe_start, end=roe_end, freq="ME")
    tickers   = roe_df["ticker"].unique()

    full_idx = pd.MultiIndex.from_product([tickers, all_me], names=["ticker", "date"])
    full_df  = pd.DataFrame(index=full_idx).reset_index()
    full_df  = full_df.merge(roe_df, on=["ticker", "date"], how="left")
    full_df  = full_df.sort_values(["ticker", "date"])
    full_df["roe"] = full_df.groupby("ticker")["roe"].ffill(limit=2)
    full_df = full_df.dropna(subset=["roe"])
    return full_df[["date", "ticker", "roe"]]

## 5. Re_5y - ROE 스프레드 계산 함수

In [12]:
def compute_spread(re5y_df: pd.DataFrame, roe_monthly: pd.DataFrame) -> pd.DataFrame:
    """
    re5y_df    : date | ticker | Re_5y  (월말)
    roe_monthly: date | ticker | roe    (월말)
    → 결합 후 rim_spread = ROE - Re_5y, ffill(limit=2)
    """
    merged = pd.merge(re5y_df, roe_monthly, on=["ticker", "date"], how="left")
    merged = merged.sort_values(["ticker", "date"])
    merged["roe"] = merged.groupby("ticker")["roe"].ffill(limit=2)
    merged["rim_spread"] = merged["roe"] - merged["Re_5y"]
    return merged

## 6. 시각화 함수

In [13]:
def plot_spread(df: pd.DataFrame, ticker: str) -> None:
    """
    Re_5y, ROE, rim_spread 추세를 3-패널 그래프로 표시
    """
    sub = df[df["ticker"] == ticker].dropna(subset=["Re_5y", "roe", "rim_spread"]).copy()
    if sub.empty:
        print(f"[WARN] {ticker}: 플롯 데이터 없음")
        return

    fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
    fig.suptitle(f"{ticker}  |  Re_5y  /  ROE  /  RIM Spread", fontsize=14)

    # --- Re_5y ---
    axes[0].plot(sub["date"], sub["Re_5y"] * 100, color="steelblue", linewidth=1.5)
    axes[0].set_ylabel("Re_5y (%)")
    axes[0].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
    axes[0].grid(True, alpha=0.3)

    # --- ROE ---
    axes[1].plot(sub["date"], sub["roe"] * 100, color="darkorange", linewidth=1.5)
    axes[1].set_ylabel("ROE (%)")
    axes[1].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
    axes[1].grid(True, alpha=0.3)

    # --- Spread ---
    axes[2].fill_between(
        sub["date"], sub["rim_spread"] * 100, 0,
        where=(sub["rim_spread"] >= 0), color="green", alpha=0.4, label="Positive"
    )
    axes[2].fill_between(
        sub["date"], sub["rim_spread"] * 100, 0,
        where=(sub["rim_spread"] < 0), color="red", alpha=0.4, label="Negative"
    )
    axes[2].axhline(0, color="black", linewidth=0.8)
    axes[2].set_ylabel("Spread (%)")
    axes[2].yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
    axes[2].set_xlabel("Date")
    axes[2].legend(loc="upper left")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 7. 역사적 평균 스프레드 분류 함수

In [14]:
def classify_moat(spread_df: pd.DataFrame) -> pd.DataFrame:
    """
    ticker 별 역사적 평균 스프레드 계산 후 해자(moat) 등급 분류.
    - strong_economic_moat : 평균 스프레드 >= 5%
    - middle               : 0% <= 평균 스프레드 < 5%
    - weak                 : 평균 스프레드 < 0%
    """
    avg = (
        spread_df.dropna(subset=["rim_spread"])
                 .groupby("ticker")["rim_spread"]
                 .mean()
                 .reset_index()
                 .rename(columns={"rim_spread": "avg_spread"})
    )
    avg["moat_grade"] = avg["avg_spread"].apply(
        lambda x: "strong_economic_moat" if x >= 0.05
                  else ("middle" if x >= 0.00 else "weak")
    )
    avg.sort_values("avg_spread", ascending=False, inplace=True)
    avg.reset_index(drop=True, inplace=True)
    return avg

## 8. DB 저장 함수 (Upsert)

In [15]:
SAVE_TABLE = "us_required_return_result"


def save_to_db(spread_df: pd.DataFrame, db_info: Dict, table: str = SAVE_TABLE) -> None:
    """
    ticker, date, indicator 기준 중복 방지 UPSERT.
    indicator: 'roe' 또는 'rim_spread'
    """
    conn = pymysql.connect(
        host=db_info["host"], port=db_info["port"],
        user=db_info["user"], password=db_info["password"],
        database=db_info["database"], charset="utf8mb4"
    )
    cursor = conn.cursor()

    upsert_sql = f"""
        INSERT INTO {table} (ticker, date, indicator, value)
        VALUES (%s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE value = VALUES(value);
    """

    rows = []
    for _, row in spread_df.iterrows():
        if pd.isna(row["roe"]) and pd.isna(row["rim_spread"]):
            continue
        date_str = row["date"].strftime("%Y-%m-%d")
        ticker   = row["ticker"]
        if not pd.isna(row["roe"]):
            rows.append((ticker, date_str, "roe", float(row["roe"])))
        if not pd.isna(row["rim_spread"]):
            rows.append((ticker, date_str, "rim_spread", float(row["rim_spread"])))

    try:
        cursor.executemany(upsert_sql, rows)
        conn.commit()
        print(f"[INFO] DB 저장 완료: {len(rows):,} rows → {table}")
    except Exception as e:
        conn.rollback()
        print(f"[ERROR] DB 저장 실패: {e}")
    finally:
        cursor.close()
        conn.close()

## 9. 분석 실행
### 9-1. 분석 대상 티커 및 기간 설정

In [16]:
# ── 분석할 티커 목록 설정 ──────────────────────────
# 아래 TICKERS 를 원하는 티커로 교체하거나,
# US_TICKER_LIST 를 그대로 사용하면 전체 목록이 처리됩니다.
# TICKERS = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN"]  # 예시
TICKERS = US_TICKER_LIST                             # 전체 목록

START_DATE = "2018-01-01"

print(f"[설정] 분석 티커 수: {len(TICKERS)}, 시작일: {START_DATE}")

[설정] 분석 티커 수: 2000, 시작일: 2018-01-01


### 9-2. 티커별 ROE 수신 → 스프레드 계산 → 시각화

In [ ]:
all_spread_frames = []

for ticker in TICKERS:
    print(f"\n{'='*50}\n처리 중: {ticker}")

    # ① FMP 에서 분기 ROE 수신
    roe_raw = fetch_fmp_roe(ticker)
    if roe_raw.empty:
        print(f"  [SKIP] {ticker}: ROE 데이터 없음")
        continue

    # ② ROE → 월말 ffill 보간
    roe_monthly = roe_to_monthly(roe_raw)

    # ③ DB 에서 Re_5y 월말 데이터 조회
    re5y_monthly = fetch_re5y_from_db(
        db_info, tickers=[ticker], start_date=START_DATE
    )
    if re5y_monthly.empty:
        print(f"  [SKIP] {ticker}: Re_5y 데이터 없음")
        continue

    # ④ 스프레드 계산
    spread_df = compute_spread(re5y_monthly, roe_monthly)
    print(spread_df.tail())

    # ⑤ 시각화
    plot_spread(spread_df, ticker)

    all_spread_frames.append(spread_df)

print("\n[완료] 전체 처리 종료")


처리 중: NVDA


### 9-3. 역사적 평균 스프레드 분류

In [4]:
if all_spread_frames:
    full_spread = pd.concat(all_spread_frames, ignore_index=True)

    moat_table = classify_moat(full_spread)

    print("[ 역사적 평균 스프레드 분류 ]")
    display(moat_table)

    strong = moat_table[moat_table["moat_grade"] == "strong_economic_moat"]
    middle = moat_table[moat_table["moat_grade"] == "middle"]
    weak   = moat_table[moat_table["moat_grade"] == "weak"]

    print(f"\nStrong Moat ({len(strong)}개): {strong['ticker'].tolist()}")
    print(f"Middle      ({len(middle)}개): {middle['ticker'].tolist()}")
    print(f"Weak        ({len(weak)}개): {weak['ticker'].tolist()}")
else:
    print("[INFO] 분류할 데이터가 없습니다.")

NameError: name 'all_spread_frames' is not defined

### 9-4. DB 저장

In [ ]:
if all_spread_frames:
    save_to_db(full_spread, db_info)
else:
    print("[INFO] 저장할 데이터가 없습니다.")